In [ ]:
import logging  # noqa: F401

import eradiate
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from eradiate.experiments import AtmosphereExperiment
from eradiate.units import unit_registry as ureg

import eradiate_disort as ed

xr.set_options(
    display_expand_attrs=False,
    # display_expand_coords="default",
    display_expand_data=False,
    display_expand_data_vars=False,
)
# logging.basicConfig(level=logging.DEBUG)
eradiate.set_mode("ckd")
# eradiate.set_mode("mono")

exp = AtmosphereExperiment(
    geometry={
        "type": "plane_parallel",
        "zgrid": (np.arange(0, 120.001, 1.0) * ureg.km).to(
            "m"
        ),  # must be coarse; need adaptive support?
    },
    surface={
        "type": "lambertian",
        "reflectance": 0.5,
    },
    atmosphere=None,
    # atmosphere={
    #     "type": "molecular",
    #     "absorption_data": "mycena",
    #     "phase": {"type": "isotropic"},  # add Rayleigh first
    # },
    illumination={
        "type": "directional",
        "zenith": 30.0,
        "azimuth": 0.0,
    },
    measures={  # tricky, start with TOA only
        "type": "mdistant",
        "construct": "hplane",
        "azimuth": 0.0,
        "zeniths": np.arange(-75.0, 76.0, 5.0),
        "srf": {"type": "delta", "wavelengths": [550.0]},
        # "srf": {"type": "uniform", "wmin": 525.0, "wmax": 575.0},
    },
)

In [ ]:
backend = ed.EradiateDisortBackend(verbose=True)
backend.process(exp)

mes_mu = backend._state.umu
mes_theta = np.rad2deg(np.arccos(mes_mu))
mes_phi = backend._state.phi

radiance = xr.DataArray(
    backend._state.uu[:, 0, :], coords=[("theta", mes_theta), ("phi", mes_phi)]
)
stacked_radiance = radiance.stack(direction=["phi", "theta"])
mask = stacked_radiance["phi"] == 0.0
np_radiance = stacked_radiance.values
np_theta = stacked_radiance["theta"].where(mask, -stacked_radiance["theta"]).values
order = np.argsort(np_theta)
np_radiance = np_radiance[order]
np_theta = np_theta[order]

In [ ]:
result_disort = backend.postprocess(exp).squeeze()
mask = np.isclose(result_disort["vaa"], 0.0)
result_disort["vza"] = result_disort["vza"].where(mask, -result_disort["vza"])
result_disort["vaa"] = result_disort["vaa"].where(mask, 0.0)
radiance = np.array(result_disort.values.flatten())
theta = np.array(result_disort["vza"].values.flatten())
order = np.argsort(theta)
radiance = radiance[order]
theta = theta[order]

In [ ]:
result_mitsuba = eradiate.run(exp, spp=10_000)

In [ ]:
plt.plot(
    theta,
    radiance,
    label="cdisort",
)
plt.plot(
    np.squeeze(result_mitsuba["vza"]),
    np.squeeze(result_mitsuba["radiance"]),
    label="mitsuba",
)
plt.legend()
plt.ylim([-0.01, 0.31])

In [ ]:
import eradiate.pipelines.logic as pplogic

spectral_grid = exp.spectral_grids[0]
viewing_angles = pplogic.viewing_angles(exp.measures[0].viewing_angles.m_as("deg"))
solar_angles = pplogic.extract_irradiance("ckd", exp.illumination, spectral_grid)[
    "solar_angles"
]

display(
    pplogic.gather_bitmaps(
        mode_id="ckd",
        var_name="radiance",
        var_metadata={},
        gather_variance=False,
        calculate_stokes=False,
        bitmaps=exp.measures[0].mi_results,
        viewing_angles=viewing_angles,
        solar_angles=solar_angles,
    )["radiance_raw"]
)

pplogic.aggregate_ckd_quad(
    mode_id="ckd",
    spectral_grid=spectral_grid,
    raw_data=backend.postprocess(),
    ckd_quads=exp.ckd_quads[0],
    is_variance=False,
)

In [ ]:
exp = AtmosphereExperiment(
    measures={
        "type": "mdistant",
        "construct": "grid",
        "zeniths": np.arange(0, 76, 5),
        "azimuths": [0, 180],
    }
)
eradiate.run(exp)